# Spaceship Titanic - Ensemble Learning

## Muc tieu
- Thuc hien exploratory data analysis (EDA) de hieu du lieu.
- Ap dung 3 ky thuat Ensemble: Bagging, Boosting, Stacking.
- So sanh hieu nang cac mo hinh qua accuracy, f1, precision, recall.
- Tao file submission cho cuoc thi Spaceship Titanic. Do chinh xac muc tieu tren 80%.

## Tai lieu tham khao
- Sklearn: Bagging/Boosting/Stacking API.
- XGBoost, LightGBM, CatBoost docs.


In [ ]:
# Cai dat: chay 1 lan trong terminal
# pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm catboost

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, time, os
warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-whitegrid")
sns.set_palette("husl")

from sklearn.model_selection import cross_val_score, StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import (
    RandomForestClassifier, ExtraTreesClassifier,
    AdaBoostClassifier, GradientBoostingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

print("Setup complete")


## Buoc 1: Tai du lieu

Doc 3 file tu thu muc data/:
- train.csv: du lieu huan luyen, co ket qua dung (Transported).
- test.csv: du lieu can du doan.
- sample_submission.csv: mau file de nop len Kaggle.


In [ ]:
DATA_DIR = 'data'
train_df = pd.read_csv(f'{DATA_DIR}/train.csv')
test_df  = pd.read_csv(f'{DATA_DIR}/test.csv')
sample   = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

print(f'Train shape: {train_df.shape}')
print(f'Test  shape: {test_df.shape}')
print('Columns:', train_df.columns.tolist())
print('Train dtypes:')
print(train_df.dtypes)
train_df.head(3)


## Buoc 2: Khai pha du lieu (EDA)

Muc tieu:
- Xem cot nao thieu du lieu (missing values).
- Xem ti le hanh khach duoc chuyen (Transported).
- Thong ke don gian ve do tuoi.

Bieu do se tu dong luu vao file fig_eda.png.


In [ ]:
print('=== Missing values ===')
print(train_df.isnull().sum())
print('
=== Target distribution ===')
print(train_df['Transported'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.countplot(data=train_df, x='Transported', ax=axes[0])
axes[0].set_title('Transported Distribution')
sns.histplot(data=train_df, x='Age', hue='Transported', bins=30, ax=axes[1], kde=True)
axes[1].set_title('Age by Transported')
plt.tight_layout()
plt.savefig('fig_eda.png', dpi=150)
plt.close()
print('EDA plot saved: fig_eda.png')


## Buoc 3: Xu ly du lieu (Preprocessing)

Cac buoc chinh:
- Bo cac cot khong can thiet: PassengerId, Name, Cabin.
- Tao dac trung GroupSize.
- Dien gia tri thieu: median cho numeric, "Missing" cho categorical.
- Ma hoa categorical features bang LabelEncoder.
- Chia train/val theo ti le 85/15.


In [ ]:
# Bước 3: Preprocessing - CAI TIEN cho accuracy > 80%
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

TARGET = "Transported"
y = train_df[TARGET].astype(int)
df = train_df.drop(columns=[TARGET])
df_test = test_df.copy()

# --- 1. Cabin: tách thành Deck / Num / Side ---
cabin_parts = df["Cabin"].fillna("Missing/0/M").str.split("/")
df["Deck"] = cabin_parts.str[0].fillna("Missing")
df["CabinNum"] = pd.to_numeric(cabin_parts.str[1], errors="coerce").fillna(0)
df["Side"] = cabin_parts.str[2].fillna("Missing")

cabin_parts_test = df_test["Cabin"].fillna("Missing/0/M").str.split("/")
df_test["Deck"] = cabin_parts_test.str[0].fillna("Missing")
df_test["CabinNum"] = pd.to_numeric(cabin_parts_test.str[1], errors="coerce").fillna(0)
df_test["Side"] = cabin_parts_test.str[2].fillna("Missing")

# --- 2. Spending: total_spending + log_spending ---
spend_cols = ["RoomService", "FoodCourt", "ShoppingMall", "Spa", "VRDeck"]
for c in spend_cols:
    df[c] = df[c].fillna(0)
    df_test[c] = df_test[c].fillna(0)
    df[c + "_log"] = np.log1p(df[c])
    df_test[c + "_log"] = np.log1p(df_test[c])

df["TotalSpending"] = df[spend_cols].sum(axis=1)
df_test["TotalSpending"] = df_test[spend_cols].sum(axis=1)
df["TotalSpending_log"] = np.log1p(df["TotalSpending"])
df_test["TotalSpending_log"] = np.log1p(df_test["TotalSpending"])

# --- 3. Age bins ---
bins = [0, 12, 18, 25, 35, 50, 65, 200]
labels = ["Child", "Teen", "YoungAdult", "Adult", "MiddleAge", "Senior", "Elder"]
df["AgeBin"] = pd.cut(df["Age"].fillna(df["Age"].median()), bins=bins, labels=labels).astype(str)
df_test["AgeBin"] = pd.cut(df_test["Age"].fillna(df_test["Age"].median()), bins=bins, labels=labels).astype(str)

# --- 4. GroupSize + IsAlone ---
pass_id_counts = pd.concat([df["PassengerId"], df_test["PassengerId"]])
group_counts = pass_id_counts.str.split("_").str[0].value_counts()
df["GroupSize"] = df["PassengerId"].str.split("_").str[0].map(group_counts).fillna(1)
df_test["GroupSize"] = df_test["PassengerId"].str.split("_").str[0].map(group_counts).fillna(1)
df["IsAlone"] = (df["GroupSize"] == 1).astype(int)
df_test["IsAlone"] = (df_test["GroupSize"] == 1).astype(int)

# --- 5. Drop ID / Name columns ---
drop_cols = ["PassengerId", "Name", "Cabin", "Age"] + spend_cols
df = df.drop(columns=drop_cols)
df_test = df_test.drop(columns=drop_cols)

# --- 6. Label Encode categoricals ---
cat_cols = df.select_dtypes(include=["object", "category"]).columns.tolist()
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()

for c in num_cols:
    df[c] = df[c].fillna(df[c].median())
    df_test[c] = df_test[c].fillna(df_test[c].median())
for c in cat_cols:
    df[c] = df[c].fillna("Missing")
    df_test[c] = df_test[c].fillna("Missing")

encoders = {}
for c in cat_cols:
    le = LabelEncoder()
    combined = pd.concat([df[c].astype(str), df_test[c].astype(str)])
    le.fit(combined)
    df[c] = le.transform(df[c].astype(str))
    df_test[c] = le.transform(df_test[c].astype(str))
    encoders[c] = le

# --- 7. Train/val split ---
X_train, X_val, y_train, y_val = train_test_split(df, y, test_size=0.15, random_state=42, stratify=y)
print(f"Train: {X_train.shape} | Val: {X_val.shape} | Test: {df_test.shape}")
print("Features:", sorted(df.columns.tolist()))
print(f"\nCategorical columns encoded: {cat_cols}")
print("So luong features sau khi encode:", len(df.columns))


## Buoc 4: Bagging - Bootstrap Aggregating

Mo hinh:
- Random Forest: 500 cay, max_depth=14, min_samples_split=5
- Extra Trees: 500 cay, max_depth=16, min_samples_split=4

Danh gia: 5-fold Cross-Validation + Validation Set.


In [ ]:
# Bước 4: Bagging - Random Forest & Extra Trees
results_bag = {}

rf = RandomForestClassifier(
    n_estimators=600, max_depth=14,
    min_samples_split=5, min_samples_leaf=3,
    random_state=42, n_jobs=-1
)
rf.fit(X_train, y_train)
rf_val = accuracy_score(y_val, rf.predict(X_val))
rf_cv = cross_val_score(rf, df, y, cv=5, scoring='accuracy').mean()
results_bag['RandomForest'] = {'val_acc': round(rf_val, 4), 'cv_acc': round(rf_cv, 4)}
print(f'RandomForest  val={rf_val:.4f}  cv={rf_cv:.4f}')

et = ExtraTreesClassifier(
    n_estimators=600, max_depth=16,
    min_samples_split=5, min_samples_leaf=2,
    random_state=42, n_jobs=-1
)
et.fit(X_train, y_train)
et_val = accuracy_score(y_val, et.predict(X_val))
et_cv = cross_val_score(et, df, y, cv=5, scoring='accuracy').mean()
results_bag['ExtraTrees'] = {'val_acc': round(et_val, 4), 'cv_acc': round(et_cv, 4)}
print(f'ExtraTrees    val={et_val:.4f}  cv={et_cv:.4f}')

bagging_df = pd.DataFrame(results_bag).T
bagging_df


## Buoc 5: Boosting

Mo hinh:
- XGBoost (Extreme Gradient Boosting)
- LightGBM (Light Gradient Boosting)
- CatBoost (Categorical Boosting)

Cac mo hinh boosting hoc tuan tu, moi lan sua loi cua lan truoc.


In [ ]:
# Bước 5: Boosting - XGBoost, LightGBM, CatBoost
results_boost = {}

xgb_clf = xgb.XGBClassifier(
    n_estimators=500, max_depth=7, learning_rate=0.04,
    subsample=0.85, colsample_bytree=0.85,
    random_state=42, n_jobs=-1, eval_metric="logloss"
)
xgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
xgb_val = accuracy_score(y_val, xgb_clf.predict(X_val))
xgb_cv = cross_val_score(xgb_clf, df, y, cv=5, scoring="accuracy").mean()
results_boost["XGBoost"] = {"val_acc": round(xgb_val, 4), "cv_acc": round(xgb_cv, 4)}
print(f"XGBoost       val={xgb_val:.4f}  cv={xgb_cv:.4f}")

lgb_clf = lgb.LGBMClassifier(
    n_estimators=500, max_depth=7, learning_rate=0.04,
    num_leaves=31, subsample=0.85, colsample_bytree=0.85,
    random_state=42, n_jobs=-1, verbose=-1
)
lgb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)])
lgb_val = accuracy_score(y_val, lgb_clf.predict(X_val))
lgb_cv = cross_val_score(lgb_clf, df, y, cv=5, scoring="accuracy").mean()
results_boost["LightGBM"] = {"val_acc": round(lgb_val, 4), "cv_acc": round(lgb_cv, 4)}
print(f"LightGBM      val={lgb_val:.4f}  cv={lgb_cv:.4f}")

cb_clf = CatBoostClassifier(
    n_estimators=500, max_depth=7, learning_rate=0.04,
    random_state=42, verbose=0, thread_count=-1
)
cb_clf.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)
cb_val = accuracy_score(y_val, cb_clf.predict(X_val))
cb_cv = cross_val_score(cb_clf, df, y, cv=5, scoring="accuracy").mean()
results_boost["CatBoost"] = {"val_acc": round(cb_val, 4), "cv_acc": round(cb_cv, 4)}
print(f"CatBoost      val={cb_val:.4f}  cv={cb_cv:.4f}")

boost_df = pd.DataFrame(results_boost).T
boost_df


## Buoc 6: Stacking (Stacked Generalization)

Stacking su dung ket qua du doan cua mo hinh co ban
lam dau vao cho mot meta-learner (Logistic Regression).

Base learners: RF, ExtraTrees, XGBoost, LightGBM, CatBoost
Meta learner: Logistic Regression


In [ ]:
# Bước 6: Stacking Ensemble
base_models = [
    ("rf", RandomForestClassifier(n_estimators=400, max_depth=14, random_state=42, n_jobs=-1)),
    ("et", ExtraTreesClassifier(n_estimators=400, max_depth=16, random_state=42, n_jobs=-1)),
    ("xgb", xgb.XGBClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1, eval_metric="logloss"
    )),
    ("lgb", lgb.LGBMClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        num_leaves=31, subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )),
    ("cb", CatBoostClassifier(
        n_estimators=400, max_depth=6, learning_rate=0.04,
        random_state=42, verbose=0, thread_count=-1
    )),
]

stack_clf = StackingClassifier(
    estimators=base_models,
    final_estimator=LogisticRegression(max_iter=2000, C=0.5),
    cv=5,
    n_jobs=-1,
    passthrough=False
)
stack_clf.fit(X_train, y_train)
stack_val = accuracy_score(y_val, stack_clf.predict(X_val))
stack_cv = cross_val_score(stack_clf, df, y, cv=5, scoring="accuracy").mean()

print(f"Stacking      val={stack_val:.4f}  cv={stack_cv:.4f}")
winner = max(
    [("XGBoost", xgb_val), ("LightGBM", lgb_val), ("CatBoost", cb_val)],
    key=lambda x: x[1]
)[0]
print(f"Best single booster: {winner}")


## Buoc 7: So sanh ket qua

Bang so sanh accuracy cua cac mo hinh:
- val_acc: accuracy tren validation set
- cv_acc: trung binh 5-fold cross-validation

Phuong phap: chon mo hinh co val_acc cao nhat de du doan.


In [ ]:
# Bước 7: So sanh ket qua
all_results = {**results_bag}
all_results.update(results_boost)
all_results['Stacking'] = {'val_acc': round(stack_val, 4), 'cv_acc': round(stack_cv, 4)}

results_df = pd.DataFrame(all_results).T
results_df = results_df.sort_values('cv_acc', ascending=False)
results_df['Method'] = ['Bagging', 'Bagging', 'Boosting', 'Boosting', 'Boosting', 'Stacking']
print('=== Benchmark Table ===')
print(results_df.round(4))


## Buoc 8: Du doan va tao file submission

- Chon model tot nhat theo val_acc.
- Du doan tren test_df.
- Luu file submission.csv.
- upload file nay len Kaggle.


In [ ]:
# Bước 8: Du doan va tao submission
models = {
    "RandomForest": rf,
    "ExtraTrees": et,
    "XGBoost": xgb_clf,
    "LightGBM": lgb_clf,
    "CatBoost": cb_clf,
    "Stacking": stack_clf,
}
val_scores = {name: accuracy_score(y_val, model.predict(X_val))
              for name, model in models.items()}
best_name = max(val_scores, key=val_scores.get)
best_model = models[best_name]
print("Validation scores:")
for name, score in sorted(val_scores.items(), key=lambda x: x[1], reverse=True):
    print(f"  {name}: {score:.4f}")

test_pred = best_model.predict(df_test)
submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Transported": test_pred.astype(bool)
})
submission.to_csv("submission.csv", index=False)
print(f"\nBest model: {best_name}")
print(f"Submission saved: submission.csv ({len(submission)} rows)")
print(submission.head())
